# 03 · Activation steering (method 3)

Builds contrastive steering vectors (mean activation of sycophantic minus honest answers, per layer) on the dev
split, sweeps layer and strength, and checks the best setting on the test split.

## Setup

Runtime → Change runtime type → **T4 GPU**. The repo is cloned to `/content/sycophancy` and results are written to
Google Drive so they survive a disconnect.

In [ ]:
import os
if not os.path.exists("/content/sycophancy"):
    !git clone https://github.com/namita-tj/sycophancy /content/sycophancy
%cd /content/sycophancy
!pip install -q -e ".[all]"

In [ ]:
# Gated models (Llama 3.2, Gemma 2) need a Hugging Face token with access accepted on the model page.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Keep results on Drive.
from google.colab import drive
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/sycophancy/results"
os.makedirs(RESULTS, exist_ok=True)
!rm -rf results && ln -s $RESULTS results

In [ ]:
# Evaluation data (MMLU + Perez et al.) and synthetic training data. Only needed once per runtime.
if not os.path.exists("data/eval/pressure.jsonl"):
    !bash scripts/prepare_data.sh
!wc -l data/eval/*.jsonl data/train/*.jsonl

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TAG = MODEL.split("/")[-1].lower()
DTYPE = "fp16"   # a T4 has no bf16; use bf16 on an A100 or L4
LAYERS = "6,10,14,18"   # Qwen2.5-1.5B has 28 layers; spread the choice over the middle

In [ ]:
!syco steer build --model {MODEL} --dtype {DTYPE} --data data/eval --layers {LAYERS} --out results/{TAG}-vectors.npz

In [ ]:
# Vectors are normalised to length 1, so alpha is in residual-stream units; widen the range if nothing changes.
# Negative alpha pushes away from the sycophantic direction. Note the "=": argparse needs it for negative numbers.
!syco steer sweep --model {MODEL} --dtype {DTYPE} --data data/eval --limit 150 --vectors results/{TAG}-vectors.npz --alphas=-4,-8,-16,-32 --out results/{TAG}-sweep.json

In [ ]:
import json
sweep = json.load(open(f"results/{TAG}-sweep.json"))
best = sweep["best"]
print(best)

In [ ]:
!syco eval --model {MODEL} --dtype {DTYPE} --data data/eval --split test --steer results/{TAG}-vectors.npz --layer {best["layer"]} --alpha={best["alpha"]} --out results/{TAG}-steered
!syco compare results/{TAG}-baseline results/{TAG}-steered --out results/{TAG}-steering.md
from IPython.display import Markdown
Markdown(open(f"results/{TAG}-steering.md").read())